# CS-781 Phase 1 — Architecture 3: Transformer-XL from scratch (v2, new BPE tokenizer)

**What is new compared with my first Transformer-XL notebook**
1. **My new from-scratch BPE tokenizer** (`a1_tokenization.py` + `vocab.json` + `merges.txt`) is used everywhere (ids: pad 0, bos 1, **eos 2**, unk 3).
2. **Decoding is chosen by BERTScore F1** (the real Phase-1 metric, roberta-large) instead of ROUGE-L. The old notebook picked plain greedy decoding, which loops ("The Respondents had not filed any affidavit..." repeated). The generator now also blocks non-ASCII / control tokens (test prompts are pure ASCII).
3. **More unique data** (800M tokens, ASCII-folded like the test prompts), so the run sees every token once and never repeats data; exact-duplicate documents are dropped.
4. **Optional MBR step**: several candidates per prompt, keep the "consensus" one (BERTScore between candidates). It is used only if it beats the single best decoding on held-out validation prompts, and it is written to a **separate** `submission_mbr.zip`.
5. The model and 2-GPU DDP training code are the same as in my first Transformer-XL run; the smoke test, time-box and resume logic are kept.

**Model.** 14 layers, d_model 512, 8 heads of 64, FFN 2048, relative positional attention, 256-token segments with a 256-token memory, tied embeddings, 64.2M parameters.


In [1]:
# ============================================================ 1. SETUP + CONFIG
import os, sys, re, json, time, math, shutil, subprocess, zipfile, pickle, importlib, importlib.util
from pathlib import Path
T0 = time.time()

QUICK_TEST = False            # True = tiny end-to-end check of every stage (~25 min).  False = the real run.

SESSION_HOURS     = 9.5      # wall-clock I allow this whole notebook (Kaggle hard limit is 12 h)
RESERVE_MIN       = 80       # minutes kept free AFTER training: decoding search + test generation + optional MBR + zip
PREP_TOTAL_TOKENS = 800_000_000   # unique tokens to cache (training consumes ~650M, so no data is repeated)
PREP_WEIGHTS      = dict(high_courts=0.38, supreme_court=0.25, tribunals=0.27, acts=0.10)
PREP_DEADLINE_MIN = 60
QUICK_TRAIN_SECONDS = 360
FORCE_PREP = False
FORCE_TRAIN = False

SCRIPTS = Path("/kaggle/working/txl"); SCRIPTS.mkdir(parents=True, exist_ok=True)           # code + tokenizer copy
WORK = Path("/kaggle/working/txl_quick" if QUICK_TEST else "/kaggle/working/txl_full"); WORK.mkdir(parents=True, exist_ok=True)   # data + checkpoints

def find_file(names, root="/kaggle/input"):
    for n in names:
        hits = list(Path(root).rglob(n))
        if hits: return str(hits[0])
    return None

TOKENIZER_PY = find_file(["a1_tokenization.py"])
assert TOKENIZER_PY, "attach a Kaggle dataset containing a1_tokenization.py, vocab.json and merges.txt"
TOK_SRC = Path(TOKENIZER_PY).parent
assert (TOK_SRC / "vocab.json").exists() and (TOK_SRC / "merges.txt").exists(), \
    f"vocab.json / merges.txt must be in the SAME folder as a1_tokenization.py ({TOK_SRC})"
shutil.copy(TOKENIZER_PY, SCRIPTS / "a1_tokenization.py")
(SCRIPTS / "tok").mkdir(exist_ok=True)
for f in ("vocab.json", "merges.txt"):
    shutil.copy(TOK_SRC / f, SCRIPTS / "tok" / f)
print("tokenizer files from:", TOK_SRC)

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN

def _valid_test(d):
    return isinstance(d, dict) and len(d) >= 100 and all(isinstance(v, dict) and "text" in v for v in d.values())
test_data, TEST_JSON = None, None
for p in Path("/kaggle/input").rglob("test.json"):
    try:
        d = json.load(open(p, encoding="utf-8"))
    except Exception:
        continue
    if _valid_test(d): test_data, TEST_JSON = d, str(p); break
if test_data is None:
    from huggingface_hub import hf_hub_download
    TEST_JSON = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_1_Pretraining/Test/test.json", repo_type="dataset", token=HF_TOKEN)
    test_data = json.load(open(TEST_JSON, encoding="utf-8"))
print("test json:", TEST_JSON, "|", len(test_data), "prompts")

CONFIG = dict(
    work_dir=str(WORK), scripts_dir=str(SCRIPTS), tokenizer_dir=str(SCRIPTS / "tok"),
    dataset_id="Exploration-Lab/CS-781-Capstone",
    sources={s: f"Phase_1_Pretraining/Train/train_{s}.json" for s in ["acts", "high_courts", "supreme_court", "tribunals"]},
    seed=781,
    prep=dict(total_tokens=PREP_TOTAL_TOKENS, weights=PREP_WEIGHTS, chunks_per_source=60,
              workers=min(8, (os.cpu_count() or 2) + 2), max_doc_tokens=40_000, min_chars=400,
              val_per_mille=5, val_max_tokens=4_000_000, deadline_min=PREP_DEADLINE_MIN, seed=781, local_dir=None),
    # ~64M parameters (rule: 50M-300M). Same shape that already trained stably for I; tokens, not size, are the limit in ~8 h on 2xT4.
    model=dict(vocab_size=32000, d_model=512, n_head=8, d_head=64, d_inner=2048, n_layer=14,
               dropout=0.0, dropatt=0.0, seg_len=256, mem_len=256, clamp_len=0),
    train=dict(target_batch_tokens=65536, peak_lr=6e-4, min_lr_ratio=0.1, warmup_steps=200, weight_decay=0.1,
               betas=[0.9, 0.95], clip=1.0, train_seconds=3600, eval_every_steps=400, eval_tokens=500_000,
               save_every_sec=1200, log_every=20, micro_bs_candidates=[32, 24, 16, 12, 8, 4]),
)
if QUICK_TEST:
    CONFIG["prep"].update(total_tokens=24_000_000, chunks_per_source=8, val_max_tokens=400_000, val_per_mille=30, deadline_min=15)
    CONFIG["train"].update(eval_every_steps=40, eval_tokens=100_000, save_every_sec=120, log_every=10)
CFG_PATH = WORK / "config.json"
json.dump(CONFIG, open(CFG_PATH, "w"), indent=1)
print("QUICK_TEST =", QUICK_TEST, "| config written ->", CFG_PATH)

tokenizer files from: /kaggle/input/datasets/abhishek786595/datasetfora3v2newjs


test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

test json: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json | 1000 prompts
QUICK_TEST = False | config written -> /kaggle/working/txl_full/config.json


## 2. Tokenizer (my from-scratch BPE) + sanity checks

In [2]:
sys.path.insert(0, str(SCRIPTS))
import a1_tokenization as a1
tokenizer = a1.Tokenizer.load(str(SCRIPTS / "tok"))
assert tokenizer.vocab_size == CONFIG["model"]["vocab_size"] == 32000
assert tokenizer.id_to_token[:5] == ["<pad>", "<s>", "</s>", "<unk>", "\u0100"], "this is not the new from-scratch tokenizer (wrong vocab.json?)"
assert (a1.PAD_ID, a1.BOS_ID, a1.EOS_ID, a1.UNK_ID) == (0, 1, 2, 3)
for s in ["The Court held that the appeal is dismissed.", "Section_144 of the CrPC ____ blank", "Rs. 5,000/- under Art. 21 (1)(a)"]:
    assert tokenizer.decode(tokenizer.encode(s)) == s, s
tw = [len(v["text"].split()) for v in test_data.values()]
sample = list(test_data.values())[:300]
ntok = [len(tokenizer.encode(v["text"])) for v in sample]
assert all(c.isascii() and c != "\n" for v in sample for c in v["text"]), "test prompts are expected to be plain ASCII without newlines"
rt = sum(tokenizer.decode(tokenizer.encode(v["text"])) == a1.normalize_text(v["text"]) for v in sample)
print(f"tokenizer OK | test prompts: {len(test_data)} (words min/max {min(tw)}/{max(tw)}) | BPE tokens/prompt min/mean/max "
      f"{min(ntok)}/{sum(ntok)//len(ntok)}/{max(ntok)} | tokens/word {sum(ntok)/(512*len(ntok)):.3f} | round-trip {rt}/{len(sample)}")

tokenizer OK | test prompts: 1000 (words min/max 512/512) | BPE tokens/prompt min/mean/max 554/610/892 | tokens/word 1.193 | round-trip 300/300


## 3. Data prep: range-request streaming -> ASCII-fold + normalise -> tokenise -> `uint16` cache (nothing big is ever loaded)

In [3]:
%%writefile /kaggle/working/txl/prep_data.py
"""Stream the (private, ~19 GB) corpus from the Hugging Face Hub with HTTP range requests,
normalise + tokenise ONLY a spread-out sample of it, and write compact uint16 token arrays.

The source files are JSON arrays with exactly one record per line, so we can `seek()` to any byte
offset (-> HTTP Range request), skip to the next line start, and parse records from there.
Nothing larger than one record is ever held in memory per worker.
"""
import hashlib
import importlib.util
import json
import os
import pickle
import re
import sys
import time
import unicodedata
from multiprocessing import Pool
from pathlib import Path

import numpy as np

EOS_ID = 2                                   # new tokenizer: <pad>=0 <s>=1 </s>=2 <unk>=3
_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'", "\u201c": '"', "\u201d": '"',
    "\u201e": '"', "\u00ab": '"', "\u00bb": '"', "\u2013": "-", "\u2014": "-", "\u2212": "-",
    "\u2010": "-", "\u2011": "-", "\u2012": "-", "\u2015": "-", "\u00a0": " ", "\u2022": "*",
    "\u00b7": "*", "\u2026": "...", "\u20b9": "Rs. ",
})
_CTRL = re.compile(r"[\x00-\x08\x0e-\x1f\x7f]")


def normalize_text(t):
    """Match the test prompts: plain ASCII, straight quotes, single spaces, no newlines."""
    if not t:
        return ""
    if not t.isascii():
        t = t.translate(_MAP)
        t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    t = _CTRL.sub("", t)
    return " ".join(t.split())


def get_text(rec):
    t = rec.get("text")
    if isinstance(t, str) and t.strip():
        return t
    for k in ("headnote", "content", "body", "judgment"):
        t = rec.get(k)
        if isinstance(t, str) and t.strip():
            return t
    return ""


def _parse(b):
    b = b.strip()
    if b.endswith(b","):
        b = b[:-1]
    return json.loads(b)


def iter_records(f, start, end):
    """Yield parsed records from a '[\\n{rec},\\n{rec},\\n...]' file, beginning at byte `start`,
    until the file position passes `end`.  `f` is any seekable binary file object."""
    pos = start
    f.seek(start)
    if start > 0:                          # we landed mid-line: discard the partial line
        pos += len(f.readline())
    while pos < end:
        line = f.readline()
        if not line:
            break
        pos += len(line)
        s = line.strip()
        if not s or s in (b"[", b"]") or not s.startswith(b"{"):
            continue
        try:
            yield _parse(s)
        except Exception:                  # record spanning several lines (not expected) -> accumulate
            parts, rec = [s], None
            for _ in range(500):
                nl = f.readline()
                if not nl:
                    break
                pos += len(nl)
                parts.append(nl.strip())
                try:
                    rec = _parse(b"\n".join(parts))
                    break
                except Exception:
                    continue
            if rec is not None:
                yield rec


# ---------------------------------------------------------------- worker side
_W = {}


def _init_worker(cfg):
    _W["cfg"] = cfg
    sys.path.insert(0, cfg["scripts_dir"])
    from a1_tokenization import Tokenizer          # our from-scratch byte-level BPE
    _W["tok"] = Tokenizer.load(cfg["tokenizer_dir"])
    if not cfg.get("local_dir"):
        from huggingface_hub import HfFileSystem
        _W["fs"] = HfFileSystem(token=os.environ.get("HF_TOKEN"))


def open_source(cfg, source, fs=None):
    rel = cfg["sources"][source]
    if cfg.get("local_dir"):
        return open(os.path.join(cfg["local_dir"], os.path.basename(rel)), "rb")
    return fs.open(f"datasets/{cfg['dataset_id']}/{rel}", "rb", block_size=cfg.get("block_size", 16 << 20))


def _run_job(job):
    source, idx, start, end, budget, deadline = job
    cfg, tok = _W["cfg"], _W["tok"]
    train, val = [], []
    n_tok = n_val = n_docs = 0
    seen = set()                                   # exact-duplicate filter inside one chunk
    t0 = time.time()
    with open_source(cfg, source, _W.get("fs")) as f:
        for rec in iter_records(f, start, end):
            if time.time() > deadline:
                break
            text = normalize_text(get_text(rec))
            if len(text) < cfg["min_chars"]:
                continue
            dh = hashlib.md5(text[:3000].encode()).digest()
            if dh in seen:
                continue
            seen.add(dh)
            bucket = int(hashlib.md5(text[:2000].encode()).hexdigest()[:8], 16) % 1000
            if len(tok._cache) > 1_500_000:            # bound tokenizer cache memory
                tok._cache.clear()
            ids = tok.encode(text)[: cfg["max_doc_tokens"]]
            ids.append(EOS_ID)
            arr = np.asarray(ids, dtype=np.uint16)
            n_docs += 1
            if bucket < cfg["val_per_mille"]:
                val.append(arr)
                n_val += len(arr)
            else:
                train.append(arr)
                n_tok += len(arr)
                if n_tok >= budget:
                    break
    return dict(source=source, idx=idx, train=train, val=val, n_tok=n_tok, n_val=n_val,
                n_docs=n_docs, secs=time.time() - t0)


def run_job(job):
    err = None
    for attempt in range(4):
        try:
            return _run_job(job)
        except Exception as e:                      # network hiccup -> retry the chunk
            err = repr(e)
            time.sleep(5 * (attempt + 1))
    return dict(source=job[0], idx=job[1], train=[], val=[], n_tok=0, n_val=0, n_docs=0, secs=0, error=err)


# ---------------------------------------------------------------- main
def main(cfg_path):
    cfg = json.load(open(cfg_path))
    P = cfg["prep"]
    work = Path(cfg["work_dir"])
    data_dir = work / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    pc = dict(cfg, **{k: P[k] for k in ("min_chars", "max_doc_tokens", "val_per_mille")})
    pc["local_dir"] = P.get("local_dir")
    rng = np.random.default_rng(P["seed"])
    deadline = time.time() + P["deadline_min"] * 60

    fs = None
    if not pc["local_dir"]:
        from huggingface_hub import HfFileSystem
        fs = HfFileSystem(token=os.environ.get("HF_TOKEN"))

    jobs_by_src = {}
    for src, rel in cfg["sources"].items():
        if pc["local_dir"]:
            size = os.path.getsize(os.path.join(pc["local_dir"], os.path.basename(rel)))
        else:
            size = fs.info(f"datasets/{cfg['dataset_id']}/{rel}")["size"]
        n = P["chunks_per_source"]
        target = P["total_tokens"] * P["weights"][src]
        js = []
        for i in range(n):
            lo, hi = size * i / n, size * (i + 1) / n
            need = target / n * 6.0                      # ~bytes of raw JSON needed for this chunk's token budget
            start = int(lo + rng.random() * max(0.0, (hi - lo) - need))
            js.append((src, i, start, int(hi), target / n, deadline))
        jobs_by_src[src] = js
        print(f"{src:14s} size={size/1e9:6.2f} GB  target={target/1e6:6.1f}M tokens  chunks={n}", flush=True)
    jobs = []                                   # round-robin so a deadline still covers every source
    for i in range(P["chunks_per_source"]):
        for src in jobs_by_src:
            jobs.append(jobs_by_src[src][i])

    train_docs, val_docs, per_src = [], [], {}
    t_start = time.time()
    total = 0
    with Pool(P["workers"], initializer=_init_worker, initargs=(pc,)) as pool:
        for k, res in enumerate(pool.imap_unordered(run_job, jobs), 1):
            train_docs += res["train"]
            val_docs += res["val"]
            total += res["n_tok"]
            s = per_src.setdefault(res["source"], dict(tokens=0, docs=0))
            s["tokens"] += res["n_tok"]
            s["docs"] += res["n_docs"]
            msg = f" ERROR {res['error']}" if res.get("error") else ""
            el = time.time() - t_start
            print(f"[{k:3d}/{len(jobs)}] {res['source']:13s} chunk={res['idx']:2d} docs={res['n_docs']:5d} "
                  f"tok={res['n_tok']/1e6:5.2f}M ({res['secs']:.0f}s) | total={total/1e6:7.1f}M "
                  f"{total/max(el,1):,.0f} tok/s elapsed={el/60:.1f}m{msg}", flush=True)

    order = rng.permutation(len(train_docs))            # document-level shuffle
    n_total = sum(len(a) for a in train_docs)
    out = np.empty(n_total, dtype=np.uint16)
    p = 0
    for i in order:
        a = train_docs[i]
        out[p:p + len(a)] = a
        p += len(a)
    np.save(data_dir / "train_tokens.npy", out)

    vorder = rng.permutation(len(val_docs))
    keep, vt = [], 0
    for i in vorder:
        if vt >= P["val_max_tokens"]:
            break
        keep.append(val_docs[i])
        vt += len(val_docs[i])
    with open(data_dir / "val_docs.pkl", "wb") as fh:
        pickle.dump(keep, fh)
    np.save(data_dir / "val_tokens.npy", np.concatenate(keep) if keep else np.zeros(0, np.uint16))
    stats = dict(train_tokens=int(n_total), train_docs=len(train_docs), val_tokens=int(vt), val_docs=len(keep),
                 per_source=per_src, minutes=(time.time() - t_start) / 60)
    json.dump(stats, open(data_dir / "prep_stats.json", "w"), indent=1)
    print("DONE", json.dumps(stats), flush=True)


if __name__ == "__main__":
    main(sys.argv[1])

Writing /kaggle/working/txl/prep_data.py


In [4]:
# Check access to the dataset + that range reads (seek into the middle of a 10 GB file) work.
import importlib, prep_data; importlib.reload(prep_data)
from huggingface_hub import HfFileSystem
fs = HfFileSystem(token=HF_TOKEN)
for s, rel in CONFIG["sources"].items():
    path = f"datasets/{CONFIG['dataset_id']}/{rel}"
    size = fs.info(path)["size"]
    with fs.open(path, "rb", block_size=16 << 20) as f:
        t0 = time.time(); r0 = next(prep_data.iter_records(f, 0, size))
        t1 = time.time(); r1 = next(prep_data.iter_records(f, size // 2, size))     # mid-file range request
    txt = prep_data.normalize_text(prep_data.get_text(r1))
    print(f"{s:14s} {size/1e9:6.2f} GB | keys={list(r0.keys())} | first rec {t1-t0:.1f}s, mid-file rec {time.time()-t1:.1f}s | mid text words={len(txt.split()):,}")
    assert len(txt) > 200, f"no usable text field found for {s}: keys={list(r1.keys())} -> extend get_text() in prep_data.py"
print("range-request streaming works")

acts             0.73 GB | keys=['title', 'category', 'year', 'url', 'text', 'word_count', 'text_length_chars'] | first rec 0.5s, mid-file rec 0.5s | mid text words=375
high_courts     10.67 GB | keys=['court', 'court_id', 'year', 'bench', 'filepath', 'page_count', 'word_count', 'char_count', 'text'] | first rec 0.4s, mid-file rec 0.6s | mid text words=331
supreme_court    1.32 GB | keys=['title', 'date', 'court', 'citations', 'bench', 'headnote', 'text', 'word_count', 'text_length_chars'] | first rec 0.6s, mid-file rec 0.8s | mid text words=2,208
tribunals        6.48 GB | keys=['text', 'word_count', 'char_count', 'title', 'date', 'year', 'tribunal'] | first rec 0.5s, mid-file rec 0.7s | mid text words=1,428
range-request streaming works


In [5]:
DATA = WORK / "data"
if (DATA / "train_tokens.npy").exists() and not FORCE_PREP:
    print("cached tokens found:", open(DATA / "prep_stats.json").read())
else:
    json.dump(CONFIG, open(CFG_PATH, "w"), indent=1)
    p = subprocess.Popen([sys.executable, "-u", str(SCRIPTS / "prep_data.py"), str(CFG_PATH)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ))
    for line in p.stdout: print(line, end="")
    p.wait(); assert p.returncode == 0, "data prep failed - re-run this cell (it is safe to repeat)"
stats = json.load(open(DATA / "prep_stats.json"))
print(f"\nTRAIN tokens {stats['train_tokens']/1e6:.1f}M in {stats['train_docs']:,} docs | VAL tokens {stats['val_tokens']/1e6:.2f}M in {stats['val_docs']:,} docs")
print("per source:", {k: f"{v['tokens']/1e6:.0f}M" for k, v in stats['per_source'].items()}, f"| prep took {stats['minutes']:.1f} min")

acts           size=  0.73 GB  target=  80.0M tokens  chunks=60
high_courts    size= 10.67 GB  target= 304.0M tokens  chunks=60
supreme_court  size=  1.32 GB  target= 200.0M tokens  chunks=60
tribunals      size=  6.48 GB  target= 216.0M tokens  chunks=60
[  1/240] acts          chunk= 0 docs=  212 tok= 1.34M (10s) | total=    1.3M 124,024 tok/s elapsed=0.2m
[  2/240] acts          chunk= 1 docs=  193 tok= 1.36M (12s) | total=    2.7M 203,172 tok/s elapsed=0.2m
[  3/240] supreme_court chunk= 0 docs=  536 tok= 3.36M (21s) | total=    6.1M 272,253 tok/s elapsed=0.4m
[  4/240] tribunals     chunk= 0 docs=  313 tok= 3.61M (26s) | total=    9.7M 356,069 tok/s elapsed=0.5m
[  5/240] acts          chunk= 2 docs=  148 tok= 1.34M (12s) | total=   11.0M 325,448 tok/s elapsed=0.6m
[  6/240] supreme_court chunk= 1 docs=  375 tok= 3.34M (26s) | total=   14.3M 387,787 tok/s elapsed=0.6m
[  7/240] tribunals     chunk= 1 docs= 1445 tok= 3.60M (26s) | total=   17.9M 452,160 tok/s elapsed=0.7m
[  8/240]

## 4. Transformer-XL model (relative attention + segment recurrence) + correctness tests

In [6]:
%%writefile /kaggle/working/txl/txl_model.py
"""Transformer-XL language model, written from scratch in PyTorch.

Implements (as in Dai et al., 2019):
  * segment-level recurrence  (hidden states of the previous segment are cached as memory,
    with no gradient flowing into them)
  * relative sinusoidal positional encoding with the u / v global biases
  * the "relative shift" trick
Design choices vs. the original paper: pre-LayerNorm (much more stable at high LR), GELU FFN,
tied input/output embeddings, batch-first tensors.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TXLConfig:
    vocab_size: int = 32000
    d_model: int = 512
    n_head: int = 8
    d_head: int = 64
    d_inner: int = 2048
    n_layer: int = 14
    dropout: float = 0.0
    dropatt: float = 0.0
    seg_len: int = 256
    mem_len: int = 256
    clamp_len: int = 0          # 0 = no clamping of relative distances


class PositionalEmbedding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0.0, d_model, 2.0) / d_model))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, pos_seq):                      # pos_seq: [K] float
        sinusoid = torch.outer(pos_seq, self.inv_freq)
        return torch.cat([sinusoid.sin(), sinusoid.cos()], dim=-1)   # [K, D]


class RelAttention(nn.Module):
    def __init__(self, cfg: TXLConfig):
        super().__init__()
        D, H, dh = cfg.d_model, cfg.n_head, cfg.d_head
        self.H, self.dh = H, dh
        self.scale = dh ** -0.5
        self.q_net = nn.Linear(D, H * dh, bias=False)        # queries only for the current segment
        self.kv_net = nn.Linear(D, 2 * H * dh, bias=False)   # keys/values for [memory | segment]
        self.r_net = nn.Linear(D, H * dh, bias=False)        # projection of relative positions
        self.o_net = nn.Linear(H * dh, D, bias=False)
        self.o_net.is_residual_out = True
        self.u = nn.Parameter(torch.zeros(H, dh))            # global content bias   (r_w_bias)
        self.v = nn.Parameter(torch.zeros(H, dh))            # global position bias  (r_r_bias)
        self.drop = nn.Dropout(cfg.dropout)
        self.dropatt = nn.Dropout(cfg.dropatt)

    @staticmethod
    def rel_shift(x):
        """x: [B,H,Q,K] where column j holds relative distance (K-1-j).  Returns the tensor with
        column j holding the distance between query i (abs. position M+i) and key j."""
        B, H, Q, K = x.shape
        x = torch.cat([x.new_zeros(B, H, Q, 1), x], dim=-1)      # [B,H,Q,K+1]
        x = x.view(B, H, K + 1, Q)
        return x[:, :, 1:].reshape(B, H, Q, K)

    def forward(self, x, mem_x, r, mask):
        # x [B,Q,D] (already layer-normed), mem_x [B,M,D] or None, r [K,D], mask [Q,K] bool (True = blocked)
        B, Q, _ = x.shape
        cat = x if mem_x is None else torch.cat([mem_x, x], dim=1)
        K = cat.size(1)
        H, dh = self.H, self.dh

        q = self.q_net(x).view(B, Q, H, dh).transpose(1, 2)                  # [B,H,Q,dh]
        kv = self.kv_net(cat).view(B, K, 2, H, dh)
        k = kv[:, :, 0].transpose(1, 2)                                      # [B,H,K,dh]
        v = kv[:, :, 1].transpose(1, 2)
        rk = self.r_net(r).view(K, H, dh).transpose(0, 1).unsqueeze(0)       # [1,H,K,dh]

        qu = (q + self.u[None, :, None, :]) * self.scale
        qv = (q + self.v[None, :, None, :]) * self.scale
        AC = torch.matmul(qu, k.transpose(-1, -2))                            # content term
        BD = self.rel_shift(torch.matmul(qv, rk.transpose(-1, -2)))           # position term
        score = (AC + BD).float().masked_fill(mask[None, None], -1e30)
        prob = self.dropatt(F.softmax(score, dim=-1)).to(v.dtype)
        out = torch.matmul(prob, v).transpose(1, 2).reshape(B, Q, H * dh)
        return self.drop(self.o_net(out))


class Block(nn.Module):
    def __init__(self, cfg: TXLConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = RelAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff1 = nn.Linear(cfg.d_model, cfg.d_inner)
        self.ff2 = nn.Linear(cfg.d_inner, cfg.d_model)
        self.ff2.is_residual_out = True
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, h, mem, r, mask):
        mem_x = None if mem is None else self.ln1(mem)
        h = h + self.attn(self.ln1(h), mem_x, r, mask)
        h = h + self.drop(self.ff2(self.drop(F.gelu(self.ff1(self.ln2(h))))))
        return h


class TransformerXL(nn.Module):
    def __init__(self, cfg: TXLConfig):
        super().__init__()
        assert cfg.d_model == cfg.n_head * cfg.d_head or True
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = PositionalEmbedding(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.out_bias = nn.Parameter(torch.zeros(cfg.vocab_size))   # output matrix = emb.weight (tied)
        self.apply(self._init)
        res_std = 0.02 / math.sqrt(2 * cfg.n_layer)
        for m in self.modules():
            if getattr(m, "is_residual_out", False):
                nn.init.normal_(m.weight, 0.0, res_std)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    @torch.no_grad()
    def _update_mems(self, hids, mems, mem_len):
        if mem_len <= 0:
            return None
        new = []
        for i, hid in enumerate(hids):
            cat = hid if mems is None else torch.cat([mems[i], hid], dim=1)
            new.append(cat[:, -mem_len:].detach().clone())
        return new

    def forward(self, ids, mems=None, labels=None, mem_len=None, logits_mode="all"):
        """ids [B,Q] -> dict(loss, logits, mems).  mems: list (one per layer) of [B,M,D] or None.
        logits_mode: 'all' | 'last' | 'none'."""
        cfg = self.cfg
        B, Q = ids.shape
        h = self.drop(self.emb(ids))
        M = 0 if mems is None else mems[0].size(1)
        K = M + Q
        pos = torch.arange(K - 1, -1, -1, device=ids.device, dtype=torch.float32)
        if cfg.clamp_len > 0:
            pos = pos.clamp(max=cfg.clamp_len)
        r = self.drop(self.pos_emb(pos))
        mask = torch.ones(Q, K, dtype=torch.bool, device=ids.device).triu(diagonal=1 + M)
        hids = []
        for i, blk in enumerate(self.blocks):
            hids.append(h)                       # input of layer i = memory for layer i next segment
            h = blk(h, None if mems is None else mems[i], r, mask)
        new_mems = self._update_mems(hids, mems, cfg.mem_len if mem_len is None else mem_len)
        h = self.ln_f(h)
        loss, logits = None, None
        if labels is not None:
            lg = F.linear(h, self.emb.weight, self.out_bias)
            loss = F.cross_entropy(lg.float().reshape(-1, cfg.vocab_size), labels.reshape(-1))
            if logits_mode == "all":
                logits = lg
        elif logits_mode == "all":
            logits = F.linear(h, self.emb.weight, self.out_bias)
        elif logits_mode == "last":
            logits = F.linear(h[:, -1:], self.emb.weight, self.out_bias)
        return {"loss": loss, "logits": logits, "mems": new_mems}

Writing /kaggle/working/txl/txl_model.py


In [7]:
import torch
importlib.reload(importlib.import_module("txl_model")) if "txl_model" in sys.modules else None
from txl_model import TXLConfig, TransformerXL

torch.manual_seed(0)
tc = TXLConfig(vocab_size=500, d_model=64, n_head=4, d_head=16, d_inner=128, n_layer=3, seg_len=16, mem_len=64)
m = TransformerXL(tc).eval()
for p_ in m.parameters():                       # large weights so attention really matters in the tests
    if p_.ndim >= 2: torch.nn.init.normal_(p_, 0, 0.25)
for b in m.blocks: b.attn.u.data.normal_(0, .5); b.attn.v.data.normal_(0, .5)
# (1) relative shift == explicit distance indexing
att = m.blocks[0].attn; B_, Q_, M_ = 2, 5, 7; K_ = M_ + Q_
qv = torch.randn(B_, 4, Q_, 16)
r = m.pos_emb(torch.arange(K_ - 1, -1, -1.0)); rk = att.r_net(r).view(K_, 4, 16).transpose(0, 1).unsqueeze(0)
BD = att.rel_shift(torch.matmul(qv, rk.transpose(-1, -2)))
R = att.r_net(m.pos_emb(torch.arange(0, K_).float())).view(K_, 4, 16).transpose(0, 1)
assert all(torch.allclose(BD[:, :, i, j], (qv[:, :, i] * R[:, M_ + i - j][None]).sum(-1), atol=1e-4)
           for i in range(Q_) for j in range(M_ + i + 1)); print("relative shift OK")
# (2) causality
x = torch.randint(0, 500, (2, 20)); y = x.clone(); y[:, 12:] = torch.randint(0, 500, (2, 8))
with torch.no_grad(): assert torch.allclose(m(x)["logits"][:, :12], m(y)["logits"][:, :12], atol=1e-4)
print("causal masking OK")
# (3) segment recurrence == one long forward pass; and memory really matters
ids = torch.randint(0, 500, (2, 40))
with torch.no_grad():
    full = m(ids)["logits"]; o1 = m(ids[:, :24], mem_len=64); o2 = m(ids[:, 24:], o1["mems"], mem_len=64)
    assert torch.allclose(full[:, 24:], o2["logits"], atol=1e-4)
    assert (full[:, 24:] - m(ids[:, 24:])["logits"]).abs().max() > 0.1
    mems = m(ids[:, :30], mem_len=64, logits_mode="none")["mems"]; outs = []
    for t_ in range(30, 40):
        o = m(ids[:, t_:t_ + 1], mems, mem_len=64, logits_mode="last"); mems = o["mems"]; outs.append(o["logits"][:, 0])
    assert torch.allclose(full[:, 30:], torch.stack(outs, 1), atol=1e-4)
print("recurrence/memory + token-by-token decoding == full forward OK")
n_params = TransformerXL(TXLConfig(**CONFIG["model"])).num_params()
print(f"model parameters: {n_params/1e6:.2f}M"); assert 50e6 <= n_params <= 300e6

relative shift OK
causal masking OK
recurrence/memory + token-by-token decoding == full forward OK
model parameters: 64.21M


## 5. Training (2-GPU DDP, time-boxed cosine schedule, resumable)

In [8]:
%%writefile /kaggle/working/txl/train_txl.py
"""Transformer-XL pre-training: torchrun-compatible DDP, fp16 autocast, time-boxed cosine schedule.

Data layout: one long uint16 token stream (documents shuffled, EOS-separated) is cut into
`world * micro_bs` contiguous lanes.  Every optimizer step each lane advances by seg_len tokens, so
the Transformer-XL memory of a lane is always the *immediately preceding* text of that lane.
The LR schedule is a function of elapsed *wall-clock* fraction, so training finishes on time no
matter how fast the GPUs turn out to be.
"""
import contextlib
import json
import math
import os
import shutil
import sys
import time
from datetime import timedelta
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from txl_model import TXLConfig, TransformerXL  # noqa: E402


def log(msg, rank, fh=None):
    if rank == 0:
        line = f"[{time.strftime('%H:%M:%S')}] {msg}"
        print(line, flush=True)
        if fh:
            fh.write(line + "\n")
            fh.flush()


def probe_micro_bs(model, mcfg, device, candidates):
    """Largest micro-batch that comfortably fits (peak < 80% of GPU memory)."""
    model.train()
    for B in candidates:
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)
            mems = None
            for _ in range(2):
                x = torch.randint(4, mcfg.vocab_size, (B, mcfg.seg_len), device=device)
                with torch.autocast("cuda", dtype=torch.float16):
                    out = model(x, mems, x)
                out["loss"].backward()
                mems = out["mems"]
            peak = torch.cuda.max_memory_allocated(device)
            total = torch.cuda.get_device_properties(device).total_memory
            model.zero_grad(set_to_none=True)
            del out, mems, x
            if peak < 0.80 * total:
                return B
        except torch.cuda.OutOfMemoryError:
            pass
        model.zero_grad(set_to_none=True)
        mems = None
        torch.cuda.empty_cache()
    return candidates[-1]


@torch.no_grad()
def evaluate(model, val, device, use_cuda, seg, lanes=8, max_tokens=600_000):
    model.eval()
    n = min(len(val), max_tokens)
    L = n // lanes
    data = torch.from_numpy(val[: lanes * L].astype(np.int64)).view(lanes, L).to(device)
    mems, tot, cnt = None, 0.0, 0
    for s in range(0, L - 1, seg):
        x = data[:, s: min(s + seg, L - 1)]
        y = data[:, s + 1: s + 1 + x.size(1)]
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_cuda):
            out = model(x, mems, y, logits_mode="none")
        mems = out["mems"]
        tot += out["loss"].item() * y.numel()
        cnt += y.numel()
    model.train()
    return tot / max(cnt, 1)


def main():
    cfg_path = os.environ.get("TXL_CONFIG", str(Path(__file__).with_name("config.json")))
    cfg = json.load(open(cfg_path))
    T = dict(cfg["train"])
    smoke = os.environ.get("TXL_SMOKE") == "1"
    work = Path(cfg["work_dir"])
    run_dir = Path(os.environ.get("TXL_RUN_DIR", work / ("smoke" if smoke else "run")))
    run_dir.mkdir(parents=True, exist_ok=True)
    if smoke:
        T.update(train_seconds=75, eval_every_steps=15, eval_tokens=40_000, save_every_sec=40, warmup_steps=5)

    rank = int(os.environ.get("RANK", 0))
    world = int(os.environ.get("WORLD_SIZE", 1))
    local = int(os.environ.get("LOCAL_RANK", 0))
    use_cuda = torch.cuda.is_available()
    if world > 1:
        dist.init_process_group("nccl" if use_cuda else "gloo", timeout=timedelta(minutes=120))
    device = torch.device(f"cuda:{local}" if use_cuda else "cpu")
    if use_cuda:
        torch.cuda.set_device(device)
    torch.manual_seed(cfg.get("seed", 781))
    fh = open(run_dir / "train_log.txt", "a") if rank == 0 else None

    mcfg = TXLConfig(**cfg["model"])
    model = TransformerXL(mcfg).to(device)
    log(f"params={model.num_params()/1e6:.2f}M world={world} device={device} smoke={smoke}", rank, fh)

    ck_path = run_dir / "last.pt"
    ck = torch.load(ck_path, map_location="cpu") if ck_path.exists() and not smoke else None
    if ck is not None:
        model.load_state_dict(ck["model"])
        micro_bs = ck["micro_bs"]
        log(f"RESUMING from step {ck['step']} (train_seconds={ck['train_seconds']:.0f})", rank, fh)
    elif use_cuda:
        micro_bs = probe_micro_bs(model, mcfg, device, T["micro_bs_candidates"])
    else:
        micro_bs = 4
    t = torch.tensor([micro_bs], device=device)
    if world > 1:
        dist.all_reduce(t, op=dist.ReduceOp.MIN)
    micro_bs = int(t.item())
    accum = max(1, round(T["target_batch_tokens"] / (world * micro_bs * mcfg.seg_len)))
    tok_per_step = world * micro_bs * mcfg.seg_len * accum
    log(f"micro_bs={micro_bs} accum={accum} tokens/step={tok_per_step:,}", rank, fh)

    decay = [p for p in model.parameters() if p.ndim >= 2]
    no_decay = [p for p in model.parameters() if p.ndim < 2]
    opt = torch.optim.AdamW(
        [{"params": decay, "weight_decay": T["weight_decay"]}, {"params": no_decay, "weight_decay": 0.0}],
        lr=T["peak_lr"], betas=tuple(T["betas"]), eps=1e-8, fused=use_cuda)
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)
    step, micro, train_secs, best_val = 0, 0, 0.0, float("inf")
    if ck is not None:
        opt.load_state_dict(ck["optimizer"])
        if ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
        step, micro, train_secs, best_val = ck["step"], ck["micro"], ck["train_seconds"], ck["best_val"]
        del ck

    net = model
    if world > 1:
        net = torch.nn.parallel.DistributedDataParallel(
            model, device_ids=[local] if use_cuda else None)

    data = np.load(work / "data" / "train_tokens.npy")
    val = np.load(work / "data" / "val_tokens.npy")
    seg = mcfg.seg_len
    lanes_total = world * micro_bs
    Ln = len(data) // lanes_total
    steps_per_epoch = (Ln - 1) // seg
    assert steps_per_epoch > 10, "training data too small for this many lanes"
    my_starts = (np.arange(micro_bs) + rank * micro_bs) * Ln
    ar = np.arange(seg + 1)
    log(f"train tokens={len(data):,} lane_len={Ln:,} micro-steps/epoch={steps_per_epoch:,} "
        f"(1 epoch = {steps_per_epoch / accum:,.0f} optimizer steps)", rank, fh)

    def get_batch(m):
        pos = (m % steps_per_epoch) * seg
        chunk = torch.from_numpy(data[my_starts[:, None] + pos + ar[None, :]].astype(np.int64))
        chunk = chunk.to(device, non_blocking=True)
        return chunk[:, :-1].contiguous(), chunk[:, 1:].contiguous()

    def lr_at(step, p):
        warm = min(1.0, (step + 1) / T["warmup_steps"])
        cos = 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))
        return T["peak_lr"] * warm * (T["min_lr_ratio"] + (1 - T["min_lr_ratio"]) * cos)

    def save(path, final=False):
        payload = {"model": model.state_dict(), "config": cfg["model"]}
        if not final:
            payload.update(optimizer=opt.state_dict(), scaler=scaler.state_dict(), step=step, micro=micro,
                           train_seconds=train_secs, micro_bs=micro_bs, best_val=best_val)
        tmp = str(path) + ".tmp"
        torch.save(payload, tmp)
        os.replace(tmp, path)
        if not final:
            json.dump(dict(step=step, train_seconds=train_secs), open(run_dir / "progress.json", "w"))

    model.train()
    opt.zero_grad(set_to_none=True)
    mems = None
    TS = float(T["train_seconds"])
    t_seg = time.time()
    t_log, tok_log, n_log, loss_log = time.time(), 0, 0, torch.zeros((), device=device)
    t_save = time.time()
    history = []
    while True:
        elapsed = train_secs + (time.time() - t_seg)
        ctrl = torch.tensor([elapsed / TS, float(time.time() - t_save > T["save_every_sec"])], device=device)
        if world > 1:
            dist.broadcast(ctrl, 0)
        p, do_save = ctrl[0].item(), bool(ctrl[1].item())
        if p >= 1.0:
            break
        lr = lr_at(step, p)
        for g in opt.param_groups:
            g["lr"] = lr
        for a in range(accum):
            if micro % steps_per_epoch == 0:
                mems = None                                   # new pass over the data
            x, y = get_batch(micro)
            last = a == accum - 1
            ctx = net.no_sync() if (world > 1 and not last) else contextlib.nullcontext()
            with ctx:
                with torch.autocast("cuda", dtype=torch.float16, enabled=use_cuda):
                    out = net(x, mems, y, logits_mode="none")
                loss = out["loss"] / accum
                scaler.scale(loss).backward()
            mems = out["mems"]
            loss_log += loss.detach()
            micro += 1
        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), T["clip"])
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        step += 1
        tok_log += tok_per_step
        n_log += 1

        if step % T["log_every"] == 0:
            lv = loss_log.item() / max(n_log, 1)
            loss_log.zero_()
            now = time.time()
            log(f"step={step} loss={lv:.4f} ppl={math.exp(min(lv, 20)):.1f} lr={lr:.2e} gnorm={float(gnorm):.2f} "
                f"tok/s={tok_log / (now - t_log):,.0f} progress={100 * p:.1f}% "
                f"elapsed={elapsed / 3600:.2f}h", rank, fh)
            t_log, tok_log, n_log = now, 0, 0
        if step % T["eval_every_steps"] == 0:
            if rank == 0:
                vl = evaluate(model, val, device, use_cuda, seg, max_tokens=T["eval_tokens"])
                history.append(dict(step=step, val_loss=vl))
                log(f"VALID step={step} val_loss={vl:.4f} val_ppl={math.exp(min(vl, 20)):.2f}", rank, fh)
                if vl < best_val:
                    best_val = vl
                    save(run_dir / "best.pt", final=True)
            if world > 1:
                dist.barrier()
        if do_save:
            train_secs += time.time() - t_seg
            t_seg = time.time()
            if rank == 0:
                save(run_dir / "last.pt")
                log(f"checkpoint saved at step {step}", rank, fh)
            t_save = time.time()
            if world > 1:
                dist.barrier()

    train_secs += time.time() - t_seg
    if rank == 0:
        vl = evaluate(model, val, device, use_cuda, seg, max_tokens=max(T["eval_tokens"], 600_000))
        history.append(dict(step=step, val_loss=vl, final=True))
        log(f"FINAL step={step} val_loss={vl:.4f} val_ppl={math.exp(min(vl, 20)):.2f}", rank, fh)
        best_val = min(best_val, vl)
        save(run_dir / "final.pt", final=True)
        save(run_dir / "last.pt")
        json.dump(dict(step=step, train_hours=train_secs / 3600, final_val_loss=vl, history=history),
                  open(run_dir / "summary.json", "w"), indent=1)
        log("TRAINING COMPLETE", rank, fh)
    if world > 1:
        dist.barrier()
        dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing /kaggle/working/txl/train_txl.py


In [9]:
import threading
RUN = WORK / "run"
n_gpu = len([l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if "GPU" in l])
print("GPUs:", n_gpu); assert n_gpu >= 1

def run_train(env_extra, ngpu, smoke, timeout=None):
    env = dict(os.environ, TXL_CONFIG=str(CFG_PATH), PYTHONUNBUFFERED="1", **env_extra)
    if smoke: env["TXL_SMOKE"] = "1"
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={ngpu}", str(SCRIPTS / "train_txl.py")]
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timer = threading.Timer(timeout, p.kill) if timeout else None
    if timer: timer.start()
    ok = False
    for line in p.stdout:
        if "Warning" in line or "warnings.warn" in line: continue
        print(line, end="")
        ok |= "TRAINING COMPLETE" in line
    p.wait()
    if timer: timer.cancel()
    return ok and p.returncode == 0

if (RUN / "summary.json").exists() and not FORCE_TRAIN:
    print("training already finished:", open(RUN / "summary.json").read()[:300])
else:
    # --- 1) short smoke test: validates NCCL/DDP, picks batch size, shows real tokens/sec.  Falls back automatically.
    WORKING = None
    for env_extra, ng in ([({}, n_gpu), ({"NCCL_P2P_DISABLE": "1"}, n_gpu)] if n_gpu > 1 else []) + [({}, 1)]:
        shutil.rmtree(WORK / "smoke", ignore_errors=True)
        print(f"\n===== SMOKE TEST gpus={ng} env={env_extra} =====")
        if run_train(env_extra, ng, smoke=True, timeout=600): WORKING = (env_extra, ng); break
        print("smoke test failed, trying next fallback ...")
    assert WORKING, "all smoke-test variants failed - read the log above"
    shutil.rmtree(WORK / "smoke", ignore_errors=True)
    # --- 2) real run: use all remaining session time except the post-training reserve
    done_s = json.load(open(RUN / "progress.json"))["train_seconds"] if (RUN / "progress.json").exists() else 0.0
    budget = QUICK_TRAIN_SECONDS if QUICK_TEST else SESSION_HOURS * 3600 - (time.time() - T0) - RESERVE_MIN * 60
    assert budget > 300, "not enough session time left for training - raise SESSION_HOURS or lower RESERVE_MIN"
    CONFIG["train"]["train_seconds"] = done_s + budget
    json.dump(CONFIG, open(CFG_PATH, "w"), indent=1)
    print(f"\n===== REAL TRAINING: {budget/3600:.2f} h more (already trained {done_s/3600:.2f} h) on {WORKING[1]} GPU(s) =====")
    ok = run_train(WORKING[0], WORKING[1], smoke=False)
    assert ok, "training process ended early - simply re-run this cell: it resumes from run/last.pt"
print(open(RUN / "summary.json").read()[:600])

GPUs: 2

===== SMOKE TEST gpus=2 env={} =====

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[08:19:22] params=64.21M world=2 device=cuda:0 smoke=True
[08:19:24] micro_bs=32 accum=4 tokens/step=65,536
[08:19:31] train tokens=795,259,394 lane_len=12,425,928 micro-steps/epoch=48,538 (1 epoch = 12,134 optimizer steps)
[08:20:08] VALID step=15 val_loss=7.0261 val_ppl=1125.58
  return func(*args, **kwargs)
[08:20:16] checkpoint saved at step 18
[08:20:21] step=20 loss=7.9512 ppl=2839.0 lr=2.18e-04 gnorm=0.58 tok/s=26,040 progress=63.6% elapsed=0.01h
[08:20:47] VALID step=30 val_loss=6.6802 val_ppl=796.51
[08:21:06] FINAL step=30 val_loss=6.6867 val_ppl=801.70
[08:21:08] TRAINING COMPLETE

===== REAL TRAINING: 7.73 h more (already trained 0.00 h) o

## 6. Generation: choose the decoding by BERTScore on held-out pseudo-test prompts, then predict the real test set

In [10]:
%%writefile /kaggle/working/txl/txl_gen.py
"""Batched continuation generation for Transformer-XL + local proxies for the leaderboard metrics
(ROUGE-L, BLEU-4, repetition rate, BERTScore F1 with roberta-large = the Phase-1 primary metric)."""
import math
import os
import random
import sys
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from txl_model import TXLConfig, TransformerXL  # noqa: E402
import a1_tokenization as a1  # noqa: E402

PAD_ID, BOS_ID, EOS_ID, UNK_ID = a1.PAD_ID, a1.BOS_ID, a1.EOS_ID, a1.UNK_ID   # 0, 1, 2, 3
_SPACE = set(b" \t\n\r\x0b\x0c")


def load_model(path, device):
    ck = torch.load(path, map_location="cpu")
    model = TransformerXL(TXLConfig(**ck["config"]))
    model.load_state_dict(ck["model"])
    return model.to(device).eval()


def byte_table(tok):
    """token id -> the raw bytes it stands for (special tokens -> b'')."""
    return [b""] * a1.N_SPECIAL + [bytes(a1.BYTE_DECODER[c] for c in t) for t in tok.id_to_token[a1.N_SPECIAL:]]


def _gen_tables(tok, dev):
    if not hasattr(tok, "_gen_cache"):
        table = byte_table(tok)
        sws = torch.tensor([b[:1] == b" " for b in table])
        bad = [i for i, b in enumerate(table) if not (len(b) > 0 and all(32 <= x < 127 for x in b))]
        tok._gen_cache = (table, sws, torch.tensor(bad, dtype=torch.long))      # bad = specials + non-ASCII/control bytes
    table, sws, bad = tok._gen_cache
    return table, sws.to(dev), bad.to(dev)


@torch.no_grad()
def generate_batch(model, tok, prompt_ids, target_words=128, max_new=420, temperature=0.0, top_p=1.0, top_k=0,
                   rep_penalty=1.0, no_repeat_ngram=0, mem_len=None, prompt_tokens=768, seed=0):
    """prompt_ids: list of token-id lists.  Returns list of word lists (>= target_words words each when possible).
    All prompts in a batch are cut to the same length (their last `L` tokens) so the batch shares one memory."""
    dev = next(model.parameters()).device
    use_cuda = dev.type == "cuda"
    gen_rng = torch.Generator(device=dev).manual_seed(seed)
    B = len(prompt_ids)
    seg = model.cfg.seg_len
    L = min(min(len(p) for p in prompt_ids), prompt_tokens)
    ctx = torch.tensor([p[-L:] for p in prompt_ids], device=dev)
    table, starts_with_space, bad_ids = _gen_tables(tok, dev)
    V = model.cfg.vocab_size

    mems = None
    with torch.autocast("cuda", dtype=torch.float16, enabled=use_cuda):
        for s in range(0, L, seg):
            last = s + seg >= L
            out = model(ctx[:, s:s + seg], mems, mem_len=mem_len, logits_mode="last" if last else "none")
            mems = out["mems"]
    logits = out["logits"][:, -1].float()

    gen = torch.zeros(B, max_new, dtype=torch.long, device=dev)
    hist = [[] for _ in range(B)]
    maps = [dict() for _ in range(B)]
    wc, in_word, done = [0] * B, [False] * B, [False] * B
    n = no_repeat_ngram
    for t in range(max_new):
        lg = logits
        lg[:, bad_ids] = -float("inf")               # never emit specials / non-ASCII bytes / control bytes
        if t == 0:                                   # prompt ends on a word boundary -> next token starts a new word
            lg = lg.masked_fill(~starts_with_space[None], -float("inf"))
        if rep_penalty != 1.0 and t > 0:
            g = gen[:, :t]
            sc = lg.gather(1, g)
            lg = lg.scatter(1, g, torch.where(sc > 0, sc / rep_penalty, sc * rep_penalty))
        if n > 1:
            rows, cols = [], []
            for b in range(B):
                h = hist[b]
                if len(h) >= n - 1:
                    ban = maps[b].get(tuple(h[-(n - 1):]))
                    if ban:
                        rows += [b] * len(ban)
                        cols += list(ban)
            if rows:
                lg[rows, cols] = -float("inf")
        if temperature <= 0:
            nxt = lg.argmax(-1)
        else:
            lg = lg / temperature
            if top_k > 0:
                kth = torch.topk(lg, min(top_k, V), -1).values[:, -1:]
                lg = lg.masked_fill(lg < kth, -float("inf"))
            if top_p < 1.0:
                sl, idx = lg.sort(-1, descending=True)
                pr = F.softmax(sl, -1)
                sl = sl.masked_fill(pr.cumsum(-1) - pr > top_p, -float("inf"))
                lg = torch.full_like(lg, -float("inf")).scatter(-1, idx, sl)
            nxt = torch.multinomial(F.softmax(lg, -1), 1, generator=gen_rng).squeeze(1)
        gen[:, t] = nxt
        nl = nxt.tolist()
        for b in range(B):
            h = hist[b]
            h.append(nl[b])
            if n > 1 and len(h) >= n:
                maps[b].setdefault(tuple(h[-n:-1]), set()).add(h[-1])
            if not done[b]:
                for byte in table[nl[b]]:
                    sp = byte in _SPACE
                    if not sp and not in_word[b]:
                        wc[b] += 1
                    in_word[b] = not sp
                if wc[b] >= target_words + 1:        # word target_words+1 has started => word target_words is complete
                    done[b] = True
        if all(done):
            break
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_cuda):
            out = model(nxt[:, None], mems, mem_len=mem_len, logits_mode="last")
        mems = out["mems"]
        logits = out["logits"][:, -1].float()

    return [tok.decode(hist[b]).split() for b in range(B)]


def finalize(words, prompt, target_words=128):
    """Exactly target_words whitespace words; (very rare) fallback pads from the prompt tail."""
    if len(words) < target_words:
        words = words + prompt.split()[-(target_words - len(words)):]
    return " ".join(words[:target_words])


def predict(model, tok, prompts, batch_size=100, target_words=128, desc="", **gen_kw):
    """prompts: list[str]. Returns list[str] aligned with prompts."""
    ids = [tok.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(ids[i]))      # similar lengths share a batch
    out = [None] * len(prompts)
    short = 0
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        words = generate_batch(model, tok, [ids[i] for i in idx], target_words=target_words, **gen_kw)
        for i, w in zip(idx, words):
            short += len(w) < target_words
            out[i] = finalize(w, prompts[i], target_words)
    if short:
        print(f"  warning: {short} continuations were shorter than {target_words} words and were padded")
    return out


# ------------------------------------------------------------------ local proxy metrics
def _lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]


def rouge_l_f1(pred, ref):
    p, r = pred.split(), ref.split()
    if not p or not r:
        return 0.0
    l = _lcs(p, r)
    if l == 0:
        return 0.0
    pr, rc = l / len(p), l / len(r)
    return 2 * pr * rc / (pr + rc)


def bleu4(pred, ref):
    p, r = pred.split(), ref.split()
    if not p:
        return 0.0
    logp = 0.0
    for n in range(1, 5):
        pc = Counter(tuple(p[i:i + n]) for i in range(len(p) - n + 1))
        rc = Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))
        match = sum(min(c, rc[g]) for g, c in pc.items())
        tot = max(sum(pc.values()), 1)
        logp += math.log((match + (1 if n > 1 else 0)) / (tot + (1 if n > 1 else 0)) if (match or n > 1) else 1e-9)
    bp = 1.0 if len(p) > len(r) else math.exp(1 - len(r) / max(len(p), 1))
    return bp * math.exp(logp / 4)


def rep4(text):
    """fraction of repeated word 4-grams (0 = no repetition, ~1 = a loop)."""
    w = text.split()
    g = [tuple(w[i:i + 4]) for i in range(len(w) - 3)]
    return 0.0 if not g else 1.0 - len(set(g)) / len(g)


def build_val_prompts(val_docs, tok, n=100, prompt_words=512, target_words=128, seed=0):
    """Pseudo test-set from held-out documents: 512-word prompt + the true next 128 words."""
    rng = random.Random(seed)
    docs = list(val_docs)
    rng.shuffle(docs)
    prompts, refs = [], []
    for d in docs:
        words = tok.decode(d.tolist() if hasattr(d, "tolist") else d).split()
        need = prompt_words + target_words
        if len(words) < need:
            continue
        s = rng.randint(0, len(words) - need)
        prompts.append(" ".join(words[s:s + prompt_words]))
        refs.append(" ".join(words[s + prompt_words:s + need]))
        if len(prompts) >= n:
            break
    return prompts, refs


# ------------------------------------------------------------------ BERTScore (Phase-1 primary metric)
_BS = {}


def get_bert_scorer(device="cuda:0"):
    if "scorer" not in _BS:
        try:
            from bert_score import BERTScorer
        except ImportError:
            import subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bert_score"], check=True)
            from bert_score import BERTScorer
        _BS["scorer"] = BERTScorer(model_type="roberta-large", lang="en", batch_size=32, device=device)
    return _BS["scorer"]


def bertscore_f1(preds, refs, device="cuda:0"):
    """per-pair BERTScore F1 (roberta-large, layer 17, no baseline rescaling) as a list of floats."""
    _, _, f1 = get_bert_scorer(device).score(list(preds), list(refs))
    return [float(x) for x in f1]


def score(preds, refs, with_bert=True, device="cuda:0"):
    res = dict(rougeL=float(np.mean([rouge_l_f1(p, q) for p, q in zip(preds, refs)])),
               bleu4=float(np.mean([bleu4(p, q) for p, q in zip(preds, refs)])),
               rep4=float(np.mean([rep4(p) for p in preds])), bertscore=None)
    if with_bert:
        try:
            res["bertscore"] = float(np.mean(bertscore_f1(preds, refs, device)))
        except Exception as e:                           # no internet / model download failed -> fall back to ROUGE-L
            print("  (BERTScore unavailable:", repr(e)[:150], ")")
    return res


# ------------------------------------------------------------------ minimum-Bayes-risk selection
def mbr_select(cands_per_prompt, sim_fn):
    """cands_per_prompt: list over prompts of lists of K candidate strings.
    sim_fn(list_a, list_b) -> list of similarities. For every prompt return the index of the candidate
    with the highest total similarity to the other candidates (the 'consensus' continuation)."""
    a, b, index = [], [], []
    for pi, cs in enumerate(cands_per_prompt):
        for i in range(len(cs)):
            for j in range(i + 1, len(cs)):
                a.append(cs[i]); b.append(cs[j]); index.append((pi, i, j))
    sims = sim_fn(a, b)
    util = [np.zeros(len(cs)) for cs in cands_per_prompt]
    for (pi, i, j), s in zip(index, sims):
        util[pi][i] += s
        util[pi][j] += s
    return [int(np.argmax(u)) for u in util]

Writing /kaggle/working/txl/txl_gen.py


In [11]:
import torch, txl_gen as G
importlib.reload(G)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ckpt = RUN / "final.pt" if (RUN / "final.pt").exists() else (RUN / "best.pt" if (RUN / "best.pt").exists() else RUN / "last.pt")
model = G.load_model(ckpt, device); print("loaded", ckpt, f"| {model.num_params()/1e6:.1f}M params")

val_docs = pickle.load(open(WORK / "data" / "val_docs.pkl", "rb"))
N_VAL = 12 if QUICK_TEST else 160
vp, vr = G.build_val_prompts(val_docs, tokenizer, n=N_VAL, seed=7)       # 512-word prompt + true next 128 words (never trained on)
print(f"{len(vp)} held-out pseudo-test prompts")

GRID = [
    dict(temperature=0.0, rep_penalty=1.0,  no_repeat_ngram=0),                      # plain greedy (what the old notebook chose)
    dict(temperature=0.0, rep_penalty=1.1,  no_repeat_ngram=0),
    dict(temperature=0.0, rep_penalty=1.15, no_repeat_ngram=4),
    dict(temperature=0.0, rep_penalty=1.2,  no_repeat_ngram=3),
    dict(temperature=0.5, top_p=0.9,  rep_penalty=1.1,  no_repeat_ngram=4),
    dict(temperature=0.7, top_p=0.9,  rep_penalty=1.1,  no_repeat_ngram=4),
    dict(temperature=0.7, top_p=0.9,  rep_penalty=1.0,  no_repeat_ngram=4),
    dict(temperature=0.8, top_k=50,   rep_penalty=1.05, no_repeat_ngram=5),
    dict(temperature=0.9, top_p=0.95, rep_penalty=1.1,  no_repeat_ngram=4),
]
if QUICK_TEST: GRID = [GRID[0], GRID[2], GRID[5]]
results = []
for kw in GRID:
    t0 = time.time(); preds = G.predict(model, tokenizer, vp, batch_size=100, seed=0, **kw)
    sc = G.score(preds, vr, device=str(device)); sc["kw"] = kw; sc["val_preds"] = preds; results.append(sc)
    bs = "n/a" if sc["bertscore"] is None else f"{sc['bertscore']:.4f}"
    print(f"BERTScore {bs} | ROUGE-L {sc['rougeL']:.4f} BLEU-4 {sc['bleu4']:.4f} rep4 {sc['rep4']:.3f} | {time.time()-t0:4.0f}s | {kw}")
use_bert = all(r["bertscore"] is not None for r in results)
best = max(results, key=(lambda r: r["bertscore"]) if use_bert else (lambda r: r["rougeL"]))
BEST_GEN, BEST_VAL_PREDS = best["kw"], best["val_preds"]
print("\nBEST decoding config (by", "BERTScore" if use_bert else "ROUGE-L", "):", BEST_GEN)
print("sample prompt tail :", " ".join(vp[0].split()[-25:]))
print("sample prediction  :", BEST_VAL_PREDS[0][:400])
print("sample reference   :", vr[0][:400])

loaded /kaggle/working/txl_full/run/final.pt | 64.2M params
160 held-out pseudo-test prompts
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore 0.8168 | ROUGE-L 0.2157 BLEU-4 0.0856 rep4 0.602 |  120s | {'temperature': 0.0, 'rep_penalty': 1.0, 'no_repeat_ngram': 0}
BERTScore 0.8265 | ROUGE-L 0.2193 BLEU-4 0.0948 rep4 0.363 |   82s | {'temperature': 0.0, 'rep_penalty': 1.1, 'no_repeat_ngram': 0}
BERTScore 0.8384 | ROUGE-L 0.2146 BLEU-4 0.0971 rep4 0.000 |   76s | {'temperature': 0.0, 'rep_penalty': 1.15, 'no_repeat_ngram': 4}
BERTScore 0.8354 | ROUGE-L 0.2008 BLEU-4 0.0888 rep4 0.000 |   75s | {'temperature': 0.0, 'rep_penalty': 1.2, 'no_repeat_ngram': 3}
BERTScore 0.8386 | ROUGE-L 0.2178 BLEU-4 0.0962 rep4 0.000 |   77s | {'temperature': 0.5, 'top_p': 0.9, 'rep_penalty': 1.1, 'no_repeat_ngram': 4}
BERTScore 0.8378 | ROUGE-L 0.2083 BLEU-4 0.0847 rep4 0.000 |   85s | {'temperature': 0.7, 'top_p': 0.9, 'rep_penalty': 1.1, 'no_repeat_ngram': 4}
BERTScore 0.8384 | ROUGE-L 0.2138 BLEU-4 0.0872 rep4 0.000 |   72s | {'temperature': 0.7, 'top_p': 0.9, 'rep_penalty': 1.0, 'no_repeat_ngram': 4}
BERTScore 0.8377 | ROUGE-L 0.2065

In [12]:
# ============================================================ 7. PREDICT TEST SET + AUDIT + submission.zip
OUT = Path("/kaggle/working")
ids_all = list(test_data.keys())
ids_ = ids_all[:12] if QUICK_TEST else ids_all
prompts = [test_data[i]["text"] for i in ids_]

def write_submission(preds, zip_name, json_name):
    assert len(preds) == len(ids_)
    for p_ in preds:
        assert len(p_.split()) == 128 and p_.isascii(), "every continuation must have exactly 128 words"
    pred = {i: {"text": t} for i, t in zip(ids_, preds)}
    json.dump(pred, open(OUT / json_name, "w", encoding="utf-8"), ensure_ascii=False)
    with zipfile.ZipFile(OUT / zip_name, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(OUT / json_name, arcname="prediction.json")          # prediction.json directly at the root of the zip
    print(f"READY: {OUT / zip_name} ({(OUT / zip_name).stat().st_size/1e6:.2f} MB) | {len(pred)} ids | zip contents: {zipfile.ZipFile(OUT / zip_name).namelist()}")

t0 = time.time()
test_preds = G.predict(model, tokenizer, prompts, batch_size=100, seed=0, **BEST_GEN)
T_GEN = time.time() - t0
print(f"generated {len(test_preds)} continuations in {T_GEN/60:.1f} min | mean rep4 {sum(G.rep4(p) for p in test_preds)/len(test_preds):.3f}")
if QUICK_TEST:
    write_submission(test_preds, "quick_submission_DO_NOT_SUBMIT.zip", "quick_prediction.json")
else:
    assert set(ids_) == set(test_data.keys())
    write_submission(test_preds, "submission.zip", "prediction_single.json")
print("\nexample:", test_preds[0][:300])
print(f"total notebook time so far: {(time.time()-T0)/3600:.2f} h")

summary = dict(quick=QUICK_TEST, params=n_params, model=CONFIG["model"], train=CONFIG["train"], prep=json.load(open(WORK / "data" / "prep_stats.json")),
               training=json.load(open(RUN / "summary.json")), decoding_sweep=[{k: v for k, v in r.items() if k != "val_preds"} for r in results],
               selected_decoding=BEST_GEN, test_generation_minutes=T_GEN / 60)
json.dump(summary, open(OUT / "txl_experiment_summary.json", "w"), indent=1, default=str)

generated 1000 continuations in 4.9 min | mean rep4 0.000
READY: /kaggle/working/submission.zip (0.22 MB) | 1000 ids | zip contents: ['prediction.json']

example: well settled that the courts should not allow the writ petition to be entertained in spite of delay. The delay cannot be condoned on the ground of laches." 11. In view of the above, the impugned notification dated 21.08.2001 is quashed and set aside. The respondents are directed to take appropriate 
total notebook time so far: 8.46 h


### 7b. Optional: MBR (consensus) decoding - only kept if it beats the single best decoding on held-out validation prompts

In [13]:
RUN_MBR = True
MBR_K = 2 if QUICK_TEST else 6                       # sampled candidates per prompt (plus the best deterministic one)
MBR_KW = dict(temperature=0.7, top_p=0.9, rep_penalty=1.1, no_repeat_ngram=4)
MIN_GAIN = 0.0005                                    # BERTScore gain needed on validation to prefer MBR
sim_fn = lambda A, B: G.bertscore_f1(A, B, str(device))

time_left = SESSION_HOURS * 3600 - (time.time() - T0)
need_s = MBR_K * T_GEN * 1.25 + 300
if not RUN_MBR or not use_bert:
    print("MBR skipped (disabled, or BERTScore unavailable).")
elif not QUICK_TEST and time_left < need_s:
    print(f"MBR skipped: needs ~{need_s/60:.0f} min but only {time_left/60:.0f} min of the session budget remain.")
else:
    # 1) validate on the held-out pseudo-test prompts
    val_c = [BEST_VAL_PREDS] + [G.predict(model, tokenizer, vp, batch_size=100, seed=100 + s, **MBR_KW) for s in range(MBR_K)]
    per_prompt = [[c[i] for c in val_c] for i in range(len(vp))]
    sel = G.mbr_select(per_prompt, sim_fn)
    mbr_val = [per_prompt[i][s] for i, s in enumerate(sel)]
    bs_single = sum(G.bertscore_f1(BEST_VAL_PREDS, vr, str(device))) / len(vp)
    bs_mbr = sum(G.bertscore_f1(mbr_val, vr, str(device))) / len(vp)
    print(f"validation BERTScore  single-best {bs_single:.4f}  vs  MBR(K={MBR_K}+1) {bs_mbr:.4f}")
    if bs_mbr > bs_single + MIN_GAIN or QUICK_TEST:
        t0 = time.time()
        test_c = [test_preds] + [G.predict(model, tokenizer, prompts, batch_size=100, seed=100 + s, **MBR_KW) for s in range(MBR_K)]
        per_prompt = [[c[i] for c in test_c] for i in range(len(prompts))]
        sel = G.mbr_select(per_prompt, sim_fn)
        mbr_test = [per_prompt[i][s] for i, s in enumerate(sel)]
        print(f"MBR on test took {(time.time()-t0)/60:.1f} min | chose the deterministic candidate for {sum(s == 0 for s in sel)}/{len(sel)} prompts")
        write_submission(mbr_test, "quick_submission_mbr_DO_NOT_SUBMIT.zip" if QUICK_TEST else "submission_mbr.zip",
                         "quick_prediction_mbr.json" if QUICK_TEST else "prediction_mbr.json")
        print("-> submit submission.zip AND submission_mbr.zip (different predictions = two useful submissions); validation says MBR is the better one.")
    else:
        print("MBR did not help on validation -> not generated. Use submission.zip.")
print(f"total notebook time: {(time.time()-T0)/3600:.2f} h")

validation BERTScore  single-best 0.8386  vs  MBR(K=6+1) 0.8463
MBR on test took 31.5 min | chose the deterministic candidate for 274/1000 prompts
READY: /kaggle/working/submission_mbr.zip (0.22 MB) | 1000 ids | zip contents: ['prediction.json']
-> submit submission.zip AND submission_mbr.zip (different predictions = two useful submissions); validation says MBR is the better one.
total notebook time: 9.11 h


## Rules check
The model is trained from scratch with 64.2M parameters (inside the 50M–300M range), only on the official competition data, using my new from-scratch BPE tokenizer.
